### Address db from JP postal dataset

### Detail
https://www.evernote.com/shard/s25/sh/e91692b8-5a95-d7f8-8473-7ab401ff64e3/ab965a7e5a323acde54e3311fabb8b52


### Data source

```
address/jp/csv/12CHIBA.csv
...
```

### Address class definition

scripts/models/address.py
```py
class Address(MongoBase):
    __collection__ = 'address'
    __structure__ = {
        '_id': ObjectId,

        'zipcode': str,  # ex) 1020072 (no hyphen)
        'province': str,  # 4. 都道府県名　…………　半角カタカナ（コード順に掲載）　（注1）
        'city': str,  # 8. 市区町村名　…………　漢字（コード順に掲載）　（注1,2）
        'area': str,  # 9. 町域名　………………　漢字（五十音順に掲載）　（注1,2）

        'province_kana': str,  # 4. 都道府県名　…………　半角カタカナ（コード順に掲載）　（注1） *全角に変換
        'city_kana': str,  # 5. 市区町村名　…………　半角カタカナ（コード順に掲載）　（注1）*全角に変換
        'area_kana': str,  # 6. 町域名　………………　半角カタカナ（五十音順に掲載）　（注1）　*全角に変換

        'municipality_code': str,  # 1. 全国地方公共団体コード（JIS X0401、X0402）………　半角数字

        'one_area_with_multiple_zipcode': bool,  # 10. 一町域が二以上の郵便番号で表される場合の表示　（注3）　（「1」は該当、「0」は該当せず）
        'one_zipcode_with_multiple_area': bool,  # 13. 一つの郵便番号で二以上の町域を表す場合の表示　（注5）　（「1」は該当、「0」は該当せず）
        'has_chome': bool,  # 12. 丁目を有する町域の場合の表示　（「1」は該当、「0」は該当せず）

        'lat': float,
        'lon': float,

        'created': datetime.datetime,
        'updated': datetime.datetime

    }
    __required_fields__ = [
        '_id', 'zipcode', 'province', 'city',
        ]
    __default_values__ = {
    }
    __validators__ = {
        'zipcode': Validator.validate_zipcode_format(allow_hyphen=False),
    }

```

## steps

### step 0. download data

https://www.post.japanpost.jp/zipcode/dl/readme.html

*required to convert to UTF-8

place files as
```
address/jp/csv/12CHIBA.csv
address/jp/csv/13CHIBA.csv
...
```

### step 1. parse data

parse csv file
```
1. 全国地方公共団体コード（JIS X0401、X0402）………　半角数字
2. （旧）郵便番号（5桁）………………………………………　半角数字
3. 郵便番号（7桁）………………………………………　半角数字
4. 都道府県名　…………　半角カタカナ（コード順に掲載）　（注1）
5. 市区町村名　…………　半角カタカナ（コード順に掲載）　（注1）
6. 町域名　………………　半角カタカナ（五十音順に掲載）　（注1）
7. 都道府県名　…………　漢字（コード順に掲載）　（注1,2）
8. 市区町村名　…………　漢字（コード順に掲載）　（注1,2）
9. 町域名　………………　漢字（五十音順に掲載）　（注1,2）
10. 一町域が二以上の郵便番号で表される場合の表示　（注3）　（「1」は該当、「0」は該当せず）
11. 小字毎に番地が起番されている町域の表示　（注4）　（「1」は該当、「0」は該当せず）
12. 丁目を有する町域の場合の表示　（「1」は該当、「0」は該当せず）
13. 一つの郵便番号で二以上の町域を表す場合の表示　（注5）　（「1」は該当、「0」は該当せず）
14. 更新の表示（注6）（「0」は変更なし、「1」は変更あり、「2」廃止（廃止データのみ使用））
15. 変更理由　（「0」は変更なし、「1」市政・区政・町政・分区・政令指定都市施行、「2」住居表示の実施、「3」区画整理、「4」郵便区調整等、「5」訂正、「6」廃止（廃止データのみ使用））
```

### step 2. preprocess

##### 2-1. 
「9. 町域名」
```
・以下に掲載がない場合
・（次のビルを除く）
・（地階・階層不明）
・（１３階）
などを削除
```

##### 2-2.
「4. 都道府県名」「5. 市区町村名」「6. 町域名」

**半角カナを全角カナに**



### step 3. save as address collection to mongodb 

```
address = Adress({
...
})
address.save()
```

### step 4. dumps collections

```
mongodump -d citywalk -c address
```

### step 5. place dump file

```
data/address/address_jp.bson
```

## Next Step

finish get_address_ja_by_postalcode() in scripts/api/geo.py